<a href="https://colab.research.google.com/github/trisha-fernandes/Data-Analysis-Python/blob/main/GroupDNA_Trisha_16976.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project Name: GroupDNA

Name: Trisha Fernandes

Roll Number: 16976

Batch: June 2026

Date: 29th June 2026

In [68]:
# Built as part of The Unlox Academy Python project
# Took help of AI to create this project
from datetime import datetime, timedelta
with open('hostel_bois.txt', 'r', encoding='utf-8') as f:
  lines = f.read().split('\n')

# -------------------------------------------------------
# Feature 1 - The Chat Parser
# -------------------------------------------------------

all_data = [] # List of all_data
system_messages = []
media_omitted = []
deleted_messages = []

for line in lines:
  if not line.strip():
    continue

  if ' - ' not in line:
    continue
  timestamp, rest = line.split(' - ',1)

  if ', ' not in timestamp:
    continue
  date, time = timestamp.split(', ', 1)

  if ': ' not in rest:
    system_messages.append({
        "timestamp" : timestamp,
        "message" : rest
    })
    continue
  sender, message = rest.split(': ',1)

  # Creating Main Dataset after parsing
  data = {
      "timestamp" : timestamp,
      "date" : datetime.strptime(date,"%d/%m/%y").date(),
      "time" : datetime.strptime(time,"%H:%M").time(),
      "sender" : sender,
      "message" : message
  }
  all_data.append(data)

  # Calculating list of Media Omitted
  if "<Media omitted>" in message:
    media_omitted.append({
      "timestamp" : timestamp,
      "sender" : sender,
      "message" : message
    })

  # Calculating list of messages that were deleted
  if "This message was deleted" in message:
    deleted_messages.append({
      "timestamp" : timestamp,
      "sender" : sender,
      "message" : message
    })

total_messages = len(all_data)

distinct_senders = set()
for record in all_data: #defined record
  distinct_senders.add(record["sender"])

total_participants = len(distinct_senders)

print(f"Successfully parsed {total_messages} messages from {total_participants} participants. Skipped {len(system_messages)} system message. There were {len(media_omitted)} media-omitted messages and {len(deleted_messages)} deleted messages")

# -------------------------------------------------------
# Feature 2 - Group Overview
# -------------------------------------------------------

oldest_message = min(datetime.strptime(record["timestamp"],"%d/%m/%y, %H:%M") for record in all_data) # Oldest Message
latest_message = max(datetime.strptime(record["timestamp"],"%d/%m/%y, %H:%M") for record in all_data) # Latest Message
total_days = (latest_message - oldest_message).days + 1

messages_per_person = {}
for record in all_data:
  person_name = record['sender']
  if person_name not in messages_per_person:
    messages_per_person[person_name] = 0
  messages_per_person[person_name] += 1

sorted_messages = sorted(
    messages_per_person.items(),
    key=lambda item: item[1],
    reverse=True
)
# total_messages_per_person = sum(messages_per_person.values()) # Summing values from a tuple within a list

print(f"\n==================================================\nGroup Overview\n==================================================")
print(f"Group            : Hostel Bois 4ever \nPeriod           : {oldest_message.strftime('%d %B %Y')} to {latest_message.strftime('%d %B %Y')} ({total_days} days) \nTotal messages   : {total_messages} \nParticipants     : {total_participants} \n\nMESSAGES PER PERSON ")

# Calculating percentage of messages of all the people
for person, count in sorted_messages:  # Created 'person' variable with value as count in the sorted_messages function
  percentage = (count/ total_messages) * 100
  print(f"  {person} : {count} ({percentage:.1f}%)")

# Feature 3 - Most Active Day and Hour
total_messages_per_date = {}
for record in all_data:
  message_date = record['date']
  if message_date not in total_messages_per_date:
    total_messages_per_date[message_date] = 0
  total_messages_per_date[message_date] += 1

max_msg_counts_per_date = max(
    total_messages_per_date.items(),
    key=lambda item: item[1]
)

messages_by_hour = {}
for record in all_data:
  hour = record['time'].hour
  if hour not in messages_by_hour:
    messages_by_hour[hour] = 0
  messages_by_hour[hour] += 1

max_msg_counts_per_hour = max(
    messages_by_hour.items(),
    key=lambda item: item[1]
)

print(f"\n  Busiest day : {datetime.strftime(max_msg_counts_per_date[0],"%d %B %Y")} ({max_msg_counts_per_date[1]} messages)")
print(f"  Busiest hour : {max_msg_counts_per_hour[0]}:00 - {max_msg_counts_per_hour[0]+1}:00 (avg {(max_msg_counts_per_hour[1]/total_days):.0f} messages per day)")

participants = sorted(messages_per_person.keys())

# -------------------------------------------------------
# Feature 4 - Activity Heatmap (NumPy)
# -------------------------------------------------------

import numpy as np
heatmap = np.zeros((total_participants, 24), dtype=int) # Creating the heatmap

participants_index = {}
for i, person in enumerate(participants):
  participants_index[person] = i

for record in all_data:
  person = record['sender']
  hour = record['time'].hour
  row = participants_index[person]
  heatmap[row, hour] += 1


print("\nACTIVITY HEATMAP (messages by hour)")

print("        ", end="")
for hour in range(0, 24, 3):
    print(f"{hour:02}  ", end="")
print(end="\n\n")

for row, person in enumerate(participants):

    row_max = heatmap[row].max() # max count of the busiest hour of each person

    print(f"{person:<8}", end="")

    for hour in range(0, 24, 3):

        value = heatmap[row, hour]

        if row_max == 0:
            ratio = 0
        else:
            ratio = value / row_max

        if ratio <= 0.25:
            symbol = "."
        elif ratio <= 0.50:
            symbol = "░"
        elif ratio <= 0.75:
            symbol = "▒"
        else:
            symbol = "█"

        print(f'{symbol:^4}', end="")

    print()

# -------------------------------------------------------
# Feature 5 - Top Words
# -------------------------------------------------------

stop_words = {
    # Pronouns
    "i", "me", "my", "we", "us", "you", "your", "he", "him", "his",
    "she", "her", "they", "them", "their", "it", "its",
    # Articles & prepositions
    "a", "an", "the", "to", "of", "for", "in", "on", "at", "by",
    "from", "up", "with", "about", "as", "into", "than", "out",
    # Conjunctions
    "and", "or", "but", "if", "so", "that", "this", "which", "when", "then",
    # Auxiliary verbs
    "is", "am", "are", "was", "were", "be", "been", "have", "has", "had",
    "do", "did", "will", "would", "can", "could", "should", "get", "got",
    # Common fillers
    "not", "no", "also", "just", "one", "all", "more", "now", "dont",
    "cant", "wont", "didnt", "yeah", "ok", "okay", "how", "what", "who", "why"
}

word_counts = {}
for record in all_data:
  message = record['message']
  if message == "<Media omitted>":
    continue

  if message == "This message was deleted":
    continue
  message = message.lower()
  for ch in ".,!?()\"':;-_":
    message = message.replace(ch, "")
  words = message.split()
  for word in words:
    if word in stop_words:
      continue
    if word not in word_counts:
      word_counts[word] = 0
    word_counts[word] += 1

sorted_words = sorted(
  word_counts.items(),
  key=lambda item: item[1],
  reverse=True
)

print("\nTHIS GROUP'S FAVOURITE WORDS")
max_count = sorted_words[0][1]
for word, count in sorted_words[:10]:
    bar_length = int((count / max_count) * 20)
    bar = "█" * bar_length
    print(f"  {word:<12}{bar} {count}")

# -------------------------------------------------------
# Feature 6 - Response Speed & Silent Streaks
# -------------------------------------------------------

# Calculating response speed
response_times = {}
for i in range(1, len(all_data)):

  previous = all_data[i-1]
  current = all_data[i]
  if previous["sender"] == current["sender"]:
    continue
  previous_time = datetime.strptime(
    previous["timestamp"],
    "%d/%m/%y, %H:%M"
  )
  current_time = datetime.strptime(
    current["timestamp"],
    "%d/%m/%y, %H:%M"
  )
  gap = current_time - previous_time
  minutes = gap.total_seconds() / 60
  person = current["sender"]
  if person not in response_times:
    response_times[person] = []
  response_times[person].append(minutes)

average_response = {}
for person in response_times:
  average_response[person] = np.median(response_times[person])

fastest = min(average_response.items(), key=lambda item: item[1])
slowest = max(average_response.items(), key=lambda item: item[1])

def format_time(minutes):
    """Display in minutes if under 60, otherwise in hours"""
    if minutes < 60:
        return f"{minutes:.1f} minutes"
    else:
        return f"{minutes/60:.1f} hours"

print(f'\nRESPONSE PATTERNS')
print(f'  Fastest replier : {fastest[0]} (avg {format_time(fastest[1])})')
print(f'  Slowest replier : {slowest[0]} (avg {format_time(slowest[1])})')

# Calculating Silent Streaks
dates_per_person = {}
for record in all_data:
  person = record["sender"]
  date = record["date"]
  if person not in dates_per_person:
    dates_per_person[person] = set()
  dates_per_person[person].add(date)

silent_streaks = {}
for person in participants:
  current_streak = 0
  max_streak = 0
  for i in range(total_days):
    current_day = oldest_message.date() + timedelta(days=i)
    if current_day in dates_per_person[person]:
      current_streak = 0
    else:
      current_streak += 1

      if current_streak > max_streak:
        max_streak = current_streak

  silent_streaks[person] = max_streak

sorted_streaks = sorted(
silent_streaks.items(),
key=lambda item:item[1],
reverse=True
)

print("\nLONGEST SILENT STREAKS (consecutive days with zero messages)")

for person, streak in sorted_streaks:
    if streak == 1:
        print(f"  {person:<8} : {streak} day")
    else:
        print(f"  {person:<8} : {streak} days")

# -------------------------------------------------------
# Feature 7 - Personality Archetype Detection
# -------------------------------------------------------

# --- Detection functions (one per archetype) ---

def spammer_score(person, all_data):
    """Avg consecutive message burst length (back-to-back without others speaking)"""
    bursts = []
    current_burst = 1
    for i in range(1, len(all_data)):
        if all_data[i]["sender"] == person and all_data[i-1]["sender"] == person:
            current_burst += 1
        elif all_data[i-1]["sender"] == person:
            bursts.append(current_burst)
            current_burst = 1
    return sum(bursts) / len(bursts) if bursts else 0


def group_mom_score(person, all_data):
    """Count of caring keywords in messages"""
    caring_keywords = ["okay", "safe", "eat", "sleep", "take care", "are you", "please", "reminder", "drink water", "don't forget"]
    count = 0
    for record in all_data:
        if record["sender"] != person:
            continue
        msg = record["message"].lower()
        for keyword in caring_keywords:
            if keyword in msg:
                count += 1
    return count


def night_owl_score(person, all_data):
    """Percentage of messages sent between 23:00 and 04:59"""
    total = 0
    night = 0
    for record in all_data:
        if record["sender"] != person:
            continue
        total += 1
        hour = record["time"].hour
        if hour >= 23 or hour <= 4:
            night += 1
    return (night / total * 100) if total > 0 else 0


def storyteller_score(person, all_data):
    """Average words per message"""
    total_words = 0
    total_msgs = 0
    for record in all_data:
        if record["sender"] != person:
            continue
        msg = record["message"]
        if msg in ("<Media omitted>", "This message was deleted"):
            continue
        total_words += len(msg.split())
        total_msgs += 1
    return (total_words / total_msgs) if total_msgs > 0 else 0


def drama_queen_score(person, all_data):
    """Percentage of messages that are all-caps (>3 chars) OR contain 2+ exclamation marks"""
    total = 0
    dramatic = 0
    for record in all_data:
        if record["sender"] != person:
            continue
        msg = record["message"]
        if msg in ("<Media omitted>", "This message was deleted"):
            continue
        total += 1
        is_all_caps = len(msg) > 3 and msg == msg.upper() and msg.replace(" ", "").isalpha()
        has_exclamations = msg.count("!") >= 2
        if is_all_caps or has_exclamations:
            dramatic += 1
    return (dramatic / total * 100) if total > 0 else 0


def ghost_score(person, all_data, total_days, oldest_message, dates_per_person):
    """Percentage of days in the chat period with zero messages"""
    active_days = len(dates_per_person.get(person, set()))
    silent_days = total_days - active_days
    return (silent_days / total_days * 100) if total_days > 0 else 0


def comedian_score(person, all_data):
    """Count of laugh keywords as a percentage of their messages"""
    laugh_words = {"lol", "lmao", "haha", "rofl", "lmfao"}
    laugh_count = 0
    total = 0
    for record in all_data:
        if record["sender"] != person:
            continue
        msg = record["message"].lower()
        if msg in ("<media omitted>", "this message was deleted"):
            continue
        total += 1
        for word in laugh_words:
            if word in msg:
                laugh_count += 1
                break  # count once per message, not per keyword
    return (laugh_count / total * 100) if total > 0 else 0


def question_master_score(person, all_data):
    """Percentage of messages ending with a '?' character"""
    total = 0
    questions = 0
    for record in all_data:
        if record["sender"] != person:
            continue
        msg = record["message"].strip()
        if msg in ("<Media omitted>", "This message was deleted"):
            continue
        total += 1
        if msg.endswith("?"):
            questions += 1
    return (questions / total * 100) if total > 0 else 0

def archetype_label(archetype, score):
    """Returns a human-readable description of the score for each archetype"""
    if archetype == "THE SPAMMER":
        return f"(avg {score:.1f} msgs in a row)"
    elif archetype == "THE GROUP MOM":
        return f"({score:.0f} caring keywords)"
    elif archetype == "THE NIGHT OWL":
        return f"({score:.1f}% msgs after 11 PM)"
    elif archetype == "THE STORYTELLER":
        return f"(avg {score:.1f} words per msg)"
    elif archetype == "THE DRAMA QUEEN":
        return f"({score:.1f}% ALL-CAPS msgs)"
    elif archetype == "THE GHOST":
        return f"(silent {score:.0f}% of days)"
    elif archetype == "THE COMEDIAN":
        return f"({score:.1f}% msgs with laughs)"
    elif archetype == "THE QUESTION MASTER":
        return f"({score:.1f}% msgs end with ?)"
    else:
        return f"(score: {score:.1f})"

# --- Score every participant on every archetype ---

archetype_scores = {}  # { person : { archetype_name : score } }

for person in participants:
    archetype_scores[person] = {
        "THE SPAMMER"        : spammer_score(person, all_data),
        "THE GROUP MOM"      : group_mom_score(person, all_data),
        "THE NIGHT OWL"      : night_owl_score(person, all_data),
        "THE STORYTELLER"    : storyteller_score(person, all_data),
        "THE DRAMA QUEEN"    : drama_queen_score(person, all_data),
        "THE GHOST"          : ghost_score(person, all_data, total_days, oldest_message, dates_per_person),
        "THE COMEDIAN"       : comedian_score(person, all_data),
        "THE QUESTION MASTER": question_master_score(person, all_data),
    }

# --- Normalize all scores to 0-100 so archetypes are comparable ---

normalized_scores = {}

for archetype in list(archetype_scores[participants[0]].keys()):
    all_values = [archetype_scores[person][archetype] for person in participants]
    max_val = max(all_values)
    for person in participants:
        if person not in normalized_scores:
            normalized_scores[person] = {}
        normalized_scores[person][archetype] = (archetype_scores[person][archetype] / max_val * 100) if max_val > 0 else 0

# --- Assign each person their highest-scoring archetype (using normalized scores) ---

assigned_archetypes = {}

for person in participants:
    scores = normalized_scores[person]
    best_archetype = max(scores, key=lambda archetype: scores[archetype])
    assigned_archetypes[person] = best_archetype

# --- Print results (using original scores for the label, normalized only for comparison) ---

print("\nPERSONALITY ARCHETYPES")
for person in participants:
    archetype = assigned_archetypes[person]
    score     = archetype_scores[person][archetype]   # original score for display
    label     = archetype_label(archetype, score)
    print(f"  {person:<10} →  {archetype:<20} {label}")

Successfully parsed 3174 messages from 6 participants. Skipped 4 system message. There were 32 media-omitted messages and 15 deleted messages

Group Overview
Group            : Hostel Bois 4ever 
Period           : 01 April 2024 to 30 May 2024 (60 days) 
Total messages   : 3174 
Participants     : 6 

MESSAGES PER PERSON 
  Rahul : 953 (30.0%)
  Priya : 718 (22.6%)
  Neha : 635 (20.0%)
  Aman : 490 (15.4%)
  Karan : 354 (11.2%)
  Vikas : 24 (0.8%)

  Busiest day : 04 May 2024 (76 messages)
  Busiest hour : 18:00 - 19:00 (avg 4 messages per day)

ACTIVITY HEATMAP (messages by hour)
        00  03  06  09  12  15  18  21  

Aman     ▒   ▒   .   .   .   .   .   .  
Karan    .   .   .   ░   █   ▒   ▒   ░  
Neha     .   .   .   █   ▒   .   █   ░  
Priya    .   .   .   █   █   ░   ▒   ░  
Rahul    .   .   .   .   ▒   ▒   █   █  
Vikas    .   .   .   ░   ▒   ░   ▒   ░  

THIS GROUP'S FAVOURITE WORDS
  guys        ████████████████████ 318
  hai         ████████████████ 268
  today       ██████

*   **This group predominantly runs on two people:** Rahul and Priya alone account for 52.6% of all messages. Vikas, despite being a group member for the full 60 days, sent only 24 messages (0.8%) and was completely silent for 11 consecutive days. The group's dynamic is essentially driven by a core of 3-4 people while Vikas is a passive observer.
*   **The group has two completely opposite sleep/activity cultures:** The heatmap tells a clear story that Aman is sending 79.8% of his messages between 11 PM and 5 AM while Priya, Neha and Karan are most active in the morning (9 AM) to afternoon window. Rahul peaks at 6-9 PM.
*   **The group enery and dynamics:** Rahul sends an average of 4.5 messages in a row making him the spammer of the group. While Priya and Neha seem to reply in two different fashions. Priya is more caring and considerate (the Group Mom) while Neha brings more flavour to the chats her all caps messages (The drama queen).




